# Lab 5 — POS Tagging with an RNN (PyTorch)

**Goals:** load CoNLL-U data, build vocabularies, create a PyTorch Dataset/DataLoader with padding, implement a simple RNN model for token classification, train and evaluate.

Save this notebook and run cells in order. If files are missing, the code will print helpful warnings.


In [ ]:
## Imports & utils
import os
import glob
from typing import List, Tuple
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


In [ ]:
## Task 1: Load and preprocess CoNLL-U files
def load_conllu(path: str) -> List[List[Tuple[str,str]]]:
    """Read a CoNLL-U file and return list of sentences; each sentence is list of (word, upos)."""
    sentences = []
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, 'r', encoding='utf-8') as f:
        sent = []
        for line in f:
            line = line.strip()
            if line == '':
                if sent:
                    sentences.append(sent)
                    sent = []
                continue
            if line.startswith('#'):
                continue
            parts = line.split('\t')
            if len(parts) < 4:
                continue
            # CoNLL-U: FORM is col 2 (index 1), UPOS is col 4 (index 3)
            token = parts[1]
            upos = parts[3]
            # skip multiword tokens (IDs with '-') and empty nodes (IDs with '.')
            id_field = parts[0]
            if '-' in id_field or '.' in id_field:
                continue
            sent.append((token, upos))
        # last sentence
        if sent:
            sentences.append(sent)
    return sentences

# Try to locate files
base_dir = 'data/UD_English-EWT'
train_path = os.path.join(base_dir, 'en_ewt-ud-train.conllu')
dev_path = os.path.join(base_dir, 'en_ewt-ud-dev.conllu')
test_path = os.path.join(base_dir, 'en_ewt-ud-test.conllu')

found = True
for p in [train_path, dev_path, test_path]:
    if not os.path.exists(p):
        print('Warning: not found', p)
        found = False

if found:
    train_sents = load_conllu(train_path)
    dev_sents = load_conllu(dev_path)
    test_sents = load_conllu(test_path)
    print('Loaded:', len(train_sents), 'train sentences,', len(dev_sents), 'dev sentences,', len(test_sents), 'test sentences')
else:
    print('\nFiles not present. If you have CoNLL-U files, place them under data/UD_English-EWT/ with names:')
    print('  en_ewt-ud-train.conllu, en_ewt-ud-dev.conllu, en_ewt-ud-test.conllu')


In [ ]:
## Build vocabularies (from train)
def build_vocabs(sentences):
    word_to_ix = {'<PAD>':0, '<UNK>':1}
    tag_to_ix = {'<PAD>':0}
    widx = 2
    tidx = 1
    for sent in sentences:
        for word, tag in sent:
            w = word
            if w not in word_to_ix:
                word_to_ix[w] = widx
                widx += 1
            if tag not in tag_to_ix:
                tag_to_ix[tag] = tidx
                tidx += 1
    return word_to_ix, tag_to_ix

if 'train_sents' in globals():
    word_to_ix, tag_to_ix = build_vocabs(train_sents)
    print('Vocab sizes — words:', len(word_to_ix), 'tags:', len(tag_to_ix))
else:
    print('Train sentences not loaded; cannot build vocabs yet')


In [ ]:
## Task 2: Dataset and DataLoader with collate_fn
class POSDataset(Dataset):
    def __init__(self, sentences, word_to_ix, tag_to_ix):
        self.sentences = sentences
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self, idx):
        sent = self.sentences[idx]
        words = [w for w,t in sent]
        tags = [t for w,t in sent]
        # map to indices
        w_idxs = [self.word_to_ix.get(w, self.word_to_ix['<UNK>']) for w in words]
        t_idxs = [self.tag_to_ix[t] for t in tags]
        return torch.LongTensor(w_idxs), torch.LongTensor(t_idxs)

def collate_fn(batch):
    # batch: list of (word_idxs_tensor, tag_idxs_tensor)
    words = [item[0] for item in batch]
    tags = [item[1] for item in batch]
    words_padded = pad_sequence(words, batch_first=True, padding_value=0)
    tags_padded = pad_sequence(tags, batch_first=True, padding_value=0)
    lengths = torch.LongTensor([len(x) for x in words])
    return words_padded, tags_padded, lengths

if 'train_sents' in globals():
    train_dataset = POSDataset(train_sents, word_to_ix, tag_to_ix)
    dev_dataset = POSDataset(dev_sents, word_to_ix, tag_to_ix)
    test_dataset = POSDataset(test_sents, word_to_ix, tag_to_ix)
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    print('DataLoaders ready — example batch shapes:')
    wb, tb, lb = next(iter(train_loader))
    print('words:', wb.shape, 'tags:', tb.shape, 'lengths:', lb.shape)
else:
    print('Datasets not prepared (data missing).')


In [ ]:
## Task 3: Simple RNN model for token classification
class SimpleRNNForTokenClassification(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, padding_idx=0, num_layers=1, bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)
        rnn_output_dim = hidden_dim * (2 if bidirectional else 1)
        self.fc = nn.Linear(rnn_output_dim, num_labels)
    def forward(self, input_ids):
        # input_ids: (batch, seq_len)
        emb = self.embedding(input_ids)  # (batch, seq_len, emb)
        rnn_out, _ = self.rnn(emb)       # (batch, seq_len, hidden)
        logits = self.fc(rnn_out)        # (batch, seq_len, num_labels)
        return logits

# Quick sanity check if vocabs exist
if 'word_to_ix' in globals():
    vocab_size = len(word_to_ix)
    num_labels = len(tag_to_ix)
    model = SimpleRNNForTokenClassification(vocab_size=vocab_size, embedding_dim=100, hidden_dim=128, num_labels=num_labels, padding_idx=0)
    print('Model instantiated. Params:', sum(p.numel() for p in model.parameters()))
else:
    print('Vocab not found; skip model init')


In [ ]:
## Task 4: Training loop
import time
def train_model(model, train_loader, dev_loader, tag_to_ix, device=None, epochs=5, lr=1e-3):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    pad_idx = tag_to_ix['<PAD>']
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    best_dev_acc = 0.0
    best_state = None
    for ep in range(1, epochs+1):
        model.train()
        total_loss = 0.0
        steps = 0
        t0 = time.time()
        for words, tags, lengths in train_loader:
            words = words.to(device)
            tags = tags.to(device)
            optimizer.zero_grad()
            logits = model(words)  # (batch, seq_len, num_labels)
            # reshape for loss: (batch*seq_len, num_labels)
            logits_flat = logits.view(-1, logits.size(-1))
            tags_flat = tags.view(-1)
            loss = criterion(logits_flat, tags_flat)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            steps += 1
        avg_loss = total_loss / max(1, steps)
        train_acc = evaluate(model, train_loader, tag_to_ix, device)
        dev_acc = evaluate(model, dev_loader, tag_to_ix, device)
        t1 = time.time()
        print(f'Epoch {ep}/{epochs} — loss: {avg_loss:.4f} — train_acc: {train_acc:.4f} — dev_acc: {dev_acc:.4f} — time: {(t1-t0):.1f}s')
        if dev_acc > best_dev_acc:
            best_dev_acc = dev_acc
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_dev_acc


In [ ]:
## Task 5: Evaluation function
def evaluate(model, data_loader, tag_to_ix, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    total_tokens = 0
    correct = 0
    pad_idx = tag_to_ix['<PAD>']
    with torch.no_grad():
        for words, tags, lengths in data_loader:
            words = words.to(device)
            tags = tags.to(device)
            logits = model(words)  # (batch, seq_len, num_labels)
            preds = torch.argmax(logits, dim=-1)
            mask = (tags != pad_idx)
            total_tokens += mask.sum().item()
            correct += ((preds == tags) & mask).sum().item()
    return correct / max(1, total_tokens)


In [ ]:
## Predict a single sentence (token-level)
def predict_sentence(model, sentence: str, word_to_ix, tag_to_ix, ix_to_tag=None, device=None, max_len=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    tokens = sentence.strip().split()
    idxs = [word_to_ix.get(t, word_to_ix['<UNK>']) for t in tokens]
    input_tensor = torch.LongTensor(idxs).unsqueeze(0)  # (1, seq_len)
    if max_len is not None:
        from torch.nn.utils.rnn import pad_sequence
        input_tensor = pad_sequence([input_tensor.squeeze(0)], batch_first=True, padding_value=0)
    input_tensor = input_tensor.to(device)
    with torch.no_grad():
        logits = model(input_tensor)
        preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().tolist()
    if ix_to_tag is None:
        ix_to_tag = {i:t for t,i in tag_to_ix.items()}
    return list(zip(tokens, [ix_to_tag.get(p, '<UNK>') for p in preds]))


In [ ]:
## Driver: train model if data is present
if 'train_loader' in globals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    vocab_size = len(word_to_ix)
    num_labels = len(tag_to_ix)
    model = SimpleRNNForTokenClassification(vocab_size=vocab_size, embedding_dim=100, hidden_dim=128, num_labels=num_labels, padding_idx=0)
    torch.manual_seed(42)
    trained_model, best_dev = train_model(model, train_loader, dev_loader, tag_to_ix, device=device, epochs=5, lr=1e-3)
    print('Best dev acc:', best_dev)
    test_acc = evaluate(trained_model, test_loader, tag_to_ix, device=device)
    print('Test accuracy:', test_acc)
    # example predictions
    ix_to_tag = {i:t for t,i in tag_to_ix.items()}
    examples = ['I love NLP', 'This is a test .', 'He went to the store .']
    for ex in examples:
        print('\nExample:', ex)
        print(predict_sentence(trained_model, ex, word_to_ix, tag_to_ix, ix_to_tag, device=device))
else:
    print('Data loaders not available; cannot train. Please place CoNLL-U files in data/UD_English-EWT/')
